In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize() 

Map(center=[8.515838945899919, -80.10966640141515], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### CHIRPS Precipitation Daily Near-Real-Time
#### Chosen precipitation dataset

In [ ]:
# Load CHIRPS daily precipitation dataset, 5566m resolution
dataset = (
    ee.ImageCollection('UCSB-CHC/CHIRPS/V3/DAILY_SAT')
    .filter(ee.Filter.date('2018-05-01', '2018-05-31'))
)

# Select precipitation band and clip each image to Panama
precipitation = dataset.select('precipitation').map(
    lambda img: img.clip(panama)
)

# Visualization parameters
precipitation_vis = {
    'min': 1.0,
    'max': 17.0,
    'palette': [
        '#001137',
        '#0aab1e',
        '#e7eb05',
        '#2c7fb8',
        '#253494'
    ],
}

# Add clipped precipitation layer
# mm/day
Map.add_layer(
    precipitation,
    precipitation_vis,
    'Panama Precipitation'
)

### Global Precipitation Measurement (GPM)nRelease 07

In [ ]:
# GPM V7 30 minute data for a single day
# date_range = ee.Date('2019-09-03').getRange('day')
date_range = ee.DateRange('2018-05-01', '2018-05-31')

# 11132 m resolution
dataset = (
    ee.ImageCollection('NASA/GPM_L3/IMERG_V07')
    .filter(ee.Filter.date(date_range))
)

# Select max precipitation and mask low values
# mm/hr
precipitation = dataset.select('precipitation').max()

mask = precipitation.gt(0.5)
precipitation = precipitation.updateMask(mask)

# Clip to Panama
precipitation_panama = precipitation.clip(panama_geom)

palette = [
    '000096', '0064ff', '00b4ff', '33db80', '9beb4a',
    'ffeb00', 'ffb300', 'ff6400', 'eb1e00', 'af0000'
]

precipitation_vis = {
    'min': 0,
    'max': 15,
    'palette': palette
}

Map.add_layer(
    precipitation_panama,
    precipitation_vis,
    'Precipitation (mm/hr)'
)

In [ ]:
# Display map
Map.centerObject(panama_geom, 7)
Map